In [1]:
%load_ext autoreload
%autoreload 2

from fee_rules import applies, in_range, capture_matches, txn_mask

In [2]:
#task 5: Which issuing country has the highest number of transactions?

import pandas as pd
df = pd.read_csv("../../data/dabstep/data/context/payments.csv")

# step 1: load
print(df.shape)                                   
counts = df["issuing_country"].value_counts()

# step 2: count per country
print(counts)           

# step 3: answer, should be NL
print(counts.idxmax())                            

(138236, 21)
issuing_country
NL    29622
IT    28329
BE    23040
SE    21716
FR    14175
LU     7171
ES     7109
GR     7074
Name: count, dtype: int64
NL


In [3]:
#task 49: What is the top country (ip_country) for fraud? A. NL, B. BE, C. ES, D. FR

f = df[df["has_fraudulent_dispute"]]
by_count = f["ip_country"].value_counts()
by_volume = f.groupby("ip_country")["eur_amount"].sum()
by_rate = by_volume / df.groupby("ip_country")["eur_amount"].sum()
print(by_count, by_volume, by_rate, sep="\n\n")

ip_country
NL    2955
BE    2493
IT    1652
SE    1627
FR     843
LU     410
ES     407
GR     378
Name: count, dtype: int64

ip_country
BE    263833.85
ES     43531.87
FR     89135.03
GR     39916.73
IT    182231.72
LU     44628.44
NL    329134.08
SE    169937.79
Name: eur_amount, dtype: float64

ip_country
BE    0.122686
ES    0.067503
FR    0.068979
GR    0.062301
IT    0.070100
LU    0.067103
NL    0.121815
SE    0.084866
Name: eur_amount, dtype: float64


In [4]:
#task 1305: For account type H and the MCC description: Eating Places and Restaurants, what would be the average fee that the card scheme GlobalCard would charge for a transaction value of 10 EUR?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
mcc_df = pd.read_csv(BASE + "merchant_category_codes.csv")
with open(BASE + "fees.json") as f:
    fees = json.load(f)

# step 1: MCC lookup
hits = mcc_df[mcc_df["description"] == "Eating Places and Restaurants"]
assert len(hits) == 1, hits
mcc = int(hits["mcc"].iloc[0])
print("MCC:", mcc)


# step 2: matching rules
matching = [
    r for r in fees
    if r["card_scheme"] == "GlobalCard"
    and applies(r["account_type"], "H")
    and applies(r["merchant_category_code"], mcc)
]
print(len(matching), sorted(r["ID"] for r in matching))

# step 3: fee per rule for a 10 EUR transaction
value = 10
per_rule = {r["ID"]: r["fixed_amount"] + r["rate"] * value / 10000 for r in matching}
print(per_rule)

# step 4: unweighted average over matching rules
avg = sum(per_rule.values()) / len(per_rule)
print(f"{avg:.6f}")
assert f"{avg:.6f}" == "0.123217"

MCC: 5812
46 [5, 38, 92, 114, 140, 141, 160, 162, 192, 204, 221, 257, 267, 276, 280, 319, 325, 357, 359, 403, 427, 428, 456, 477, 498, 513, 556, 572, 612, 660, 666, 682, 688, 704, 709, 725, 741, 792, 813, 861, 888, 891, 892, 899, 917, 921]
{5: 0.199, 38: 0.139, 92: 0.112, 114: 0.175, 140: 0.228, 141: 0.098, 160: 0.15000000000000002, 162: 0.133, 192: 0.135, 204: 0.071, 221: 0.056, 257: 0.179, 267: 0.21500000000000002, 276: 0.11499999999999999, 280: 0.096, 319: 0.185, 325: 0.109, 357: 0.081, 359: 0.074, 403: 0.114, 427: 0.21600000000000003, 428: 0.05, 456: 0.07100000000000001, 477: 0.174, 498: 0.046, 513: 0.15200000000000002, 556: 0.177, 572: 0.104, 612: 0.089, 660: 0.107, 666: 0.15500000000000003, 682: 0.114, 688: 0.179, 704: 0.103, 709: 0.137, 725: 0.019, 741: 0.078, 792: 0.10300000000000001, 813: 0.14600000000000002, 861: 0.067, 888: 0.102, 891: 0.189, 892: 0.116, 899: 0.16999999999999998, 917: 0.108, 921: 0.032}
0.123217


In [5]:
#task 1753: What are the applicable fee IDs for Belles_cookbook_store in March 2023?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
MERCHANT = "Belles_cookbook_store"

# step 1: load data
df = pd.read_csv(BASE + "payments.csv")
with open(BASE + "merchant_data.json") as f:
    merchant_data = json.load(f)
with open(BASE + "fees.json") as f:
    fees = json.load(f)

merchant = next(m for m in merchant_data if m["merchant"] == MERCHANT)
print(merchant)  # check the key names here if a KeyError appears below

# step 2: March 2023 transactions (2023 is not a leap year: March = day_of_year 60-90)
march = df[
    (df["merchant"] == MERCHANT)
    & (df["year"] == 2023)
    & (df["day_of_year"].between(60, 90))
].copy()
march["intracountry"] = march["issuing_country"] == march["acquirer_country"]
print("March transactions:", len(march))

# step 3: merchant-level attributes
account_type = merchant["account_type"]
capture_delay = str(merchant["capture_delay"])
mcc = int(merchant["merchant_category_code"])
print(account_type, capture_delay, mcc)

# step 4: monthly volume (EUR) and fraud ratio (fraud volume / total volume, in %)
monthly_volume = march["eur_amount"].sum()
fraud_volume = march.loc[march["has_fraudulent_dispute"], "eur_amount"].sum()
fraud_level = 100 * fraud_volume / monthly_volume
print(f"volume: {monthly_volume:.2f} EUR, fraud level: {fraud_level:.4f}%")

# step 5: rules matching the merchant-level conditions
merchant_rules = [
    r for r in fees
    if applies(r["account_type"], account_type)
    and applies(r["merchant_category_code"], mcc)
    and capture_matches(r["capture_delay"], capture_delay)
    and in_range(r["monthly_volume"], monthly_volume)
    and in_range(r["monthly_fraud_level"], fraud_level)
]
print(len(merchant_rules), sorted(r["ID"] for r in merchant_rules))

# step 6: keep rules that match at least one March transaction
applicable = [r["ID"] for r in merchant_rules if txn_mask(r, march).any()]
print(len(applicable), sorted(applicable))

# step 7: compare with the public answer (set comparison, order does not matter)
expected = {384, 394, 276, 150, 536, 286, 163, 36, 680, 939, 428, 813, 556, 51, 53,
            572, 960, 64, 709, 454, 595, 725, 473, 347, 477, 608, 868, 741, 231, 107,
            626, 249, 123, 381}
print("missing:", sorted(expected - set(applicable)))
print("extra:  ", sorted(set(applicable) - expected))
assert set(applicable) == expected

{'merchant': 'Belles_cookbook_store', 'capture_delay': '1', 'acquirer': ['lehman_brothers'], 'merchant_category_code': 5942, 'account_type': 'R'}
March transactions: 1277
R 1 5942
volume: 116436.32 EUR, fraud level: 10.2488%
47 [36, 51, 53, 64, 65, 80, 107, 123, 150, 163, 183, 231, 249, 276, 286, 304, 347, 381, 384, 394, 428, 454, 473, 477, 498, 536, 556, 572, 595, 608, 626, 631, 678, 680, 709, 725, 741, 813, 849, 861, 868, 871, 892, 924, 939, 942, 960]
34 [36, 51, 53, 64, 107, 123, 150, 163, 231, 249, 276, 286, 347, 381, 384, 394, 428, 454, 473, 477, 536, 556, 572, 595, 608, 626, 680, 709, 725, 741, 813, 868, 939, 960]
missing: []
extra:   []


In [6]:
# task 1871: In January 2023 what delta would Belles_cookbook_store pay
# if the relative fee of the fee with ID=384 changed to 1?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
MERCHANT = "Belles_cookbook_store"
FEE_ID = 384

# step 1: load data
df = pd.read_csv(BASE + "payments.csv")

with open(BASE + "merchant_data.json") as f:
    merchant_data = json.load(f)

with open(BASE + "fees.json") as f:
    fees = json.load(f)

merchant = next(m for m in merchant_data if m["merchant"] == MERCHANT)
fee = next(r for r in fees if r["ID"] == FEE_ID)

print("Merchant:", merchant)
print("Fee rule:", fee)


# step 2: January 2023 transactions
january = df[
    (df["merchant"] == MERCHANT)
    & (df["year"] == 2023)
    & (df["day_of_year"].between(1, 31))
].copy()

january["intracountry"] = (
    january["issuing_country"] == january["acquirer_country"]
)

print("January transactions:", len(january))


# step 3: merchant-level and monthly attributes
account_type = merchant["account_type"]
capture_delay = str(merchant["capture_delay"])
mcc = int(merchant["merchant_category_code"])

monthly_volume = january["eur_amount"].sum()

fraud_volume = january.loc[
    january["has_fraudulent_dispute"],
    "eur_amount"
].sum()

fraud_level = 100 * fraud_volume / monthly_volume

print("Account type:", account_type)
print("Capture delay:", capture_delay)
print("MCC:", mcc)
print(f"January volume: {monthly_volume:.2f} EUR")
print(f"January fraud level: {fraud_level:.4f}%")

# step 4: check whether Fee 384 matches merchant/month conditions
merchant_rules = [
    r for r in fees
    if applies(r["account_type"], account_type)
    and applies(r["merchant_category_code"], mcc)
    and capture_matches(r["capture_delay"], capture_delay)
    and in_range(r["monthly_volume"], monthly_volume)
    and in_range(r["monthly_fraud_level"], fraud_level)
]

matching_ids = sorted(r["ID"] for r in merchant_rules)

print("Merchant/month matching rules:", len(matching_ids))
print(matching_ids)

fee384_merchant_match = FEE_ID in matching_ids
print(
    "Fee 384 matches merchant/month conditions:",
    fee384_merchant_match
)

# The question explicitly names Fee 384
fee384 = fee

# step 5: find January transactions to which Fee 384 applies
mask = txn_mask(fee384, january)
affected = january.loc[mask].copy()

print("Fee 384 affected transactions:", len(affected))
print(
    "Affected transaction volume:",
    affected["eur_amount"].sum()
)


# step 6: calculate original and counterfactual fees
old_rate = fee384["rate"]
new_rate = 1

fixed_amount = fee384["fixed_amount"]

affected["old_fee"] = (
    fixed_amount
    + old_rate * affected["eur_amount"] / 10000
)

affected["new_fee"] = (
    fixed_amount
    + new_rate * affected["eur_amount"] / 10000
)

old_total = affected["old_fee"].sum()
new_total = affected["new_fee"].sum()

delta = new_total - old_total

print("Original rate:", old_rate)
print("New rate:", new_rate)
print(f"Original total fee: {old_total:.14f}")
print(f"New total fee: {new_total:.14f}")
print(f"Delta: {delta:.14f}")


# cross-check: fixed_amount cancels out
affected_volume = affected["eur_amount"].sum()

delta_closed_form = (
    (new_rate - old_rate)
    * affected_volume
    / 10000
)

print(f"Delta (closed form): {delta_closed_form:.14f}")

assert abs(delta - delta_closed_form) < 1e-9


# step 7: compare with public answer
expected = -0.94810300000017

print(f"Expected: {expected:.14f}")

assert abs(delta - expected) < 1e-9

Merchant: {'merchant': 'Belles_cookbook_store', 'capture_delay': '1', 'acquirer': ['lehman_brothers'], 'merchant_category_code': 5942, 'account_type': 'R'}
Fee rule: {'ID': 384, 'card_scheme': 'NexPay', 'account_type': [], 'capture_delay': None, 'monthly_fraud_level': None, 'monthly_volume': None, 'merchant_category_code': [], 'is_credit': True, 'aci': ['C', 'B'], 'fixed_amount': 0.05, 'rate': 14, 'intracountry': None}
January transactions: 1201
Account type: R
Capture delay: 1
MCC: 5942
January volume: 113260.42 EUR
January fraud level: 10.3131%
Merchant/month matching rules: 47
[36, 51, 53, 64, 65, 80, 107, 123, 150, 163, 183, 231, 249, 276, 286, 304, 347, 381, 384, 394, 428, 454, 473, 477, 498, 536, 556, 572, 595, 608, 626, 631, 678, 680, 709, 725, 741, 813, 849, 861, 868, 871, 892, 924, 939, 942, 960]
Fee 384 matches merchant/month conditions: True
Fee 384 affected transactions: 12
Affected transaction volume: 729.3100000000001
Original rate: 14
New rate: 1
Original total fee: 1.62

In [13]:
# task 2697 diagnostic 2: aggregate-volume hypothesis
# treat the total fraud volume as ONE transaction value, as task 1305 did with 10 EUR

V = fraud["eur_amount"].sum()
schemes_present = set(fraud["card_scheme"].unique())
credit_present = set(fraud["is_credit"].unique())
print("Fraud volume:", V, "| schemes:", schemes_present, "| is_credit values:", credit_present)


def rules_for(aci, use_scheme=False, use_credit=False):
    rs = [r for r in merchant_rules if applies(r["aci"], aci)]
    if use_scheme:
        rs = [r for r in rs if r["card_scheme"] is None or r["card_scheme"] in schemes_present]
    if use_credit:
        rs = [r for r in rs if r["is_credit"] is None or r["is_credit"] in credit_present]
    return rs


variants = {
    "ACI only": (False, False),
    "ACI + scheme": (True, False),
    "ACI + is_credit": (False, True),
    "ACI + scheme + is_credit": (True, True),
}

rows = []
for name, (us, uc) in variants.items():
    totals = {}
    for aci in CANDIDATE_ACIS:
        fs = [r["fixed_amount"] + r["rate"] * V / 10000 for r in rules_for(aci, us, uc)]
        if fs:
            totals[aci] = {"mean": sum(fs) / len(fs), "min": min(fs), "n_rules": len(fs)}
    for mode in ("mean", "min"):
        vals = {a: t[mode] for a, t in totals.items()}
        best = min(vals, key=vals.get)
        rows.append((name, mode, best, round(vals[best], 2), round(vals.get("E", float("nan")), 2),
                     totals.get("E", {}).get("n_rules", 0)))

out2 = pd.DataFrame(rows, columns=["variant", "mode", "best_aci", "best_total", "E_total", "E_rules"])
print(out2.to_string(index=False))
print("\nMatches E:13.57:")
print(out2[(out2["best_aci"] == "E") & (out2["best_total"] == 13.57)])

Fraud volume: 11680.62 | schemes: {'GlobalCard', 'SwiftCharge', 'TransactPlus', 'NexPay'} | is_credit values: {np.True_}
                 variant mode best_aci  best_total  E_total  E_rules
                ACI only mean        C       54.28    60.63        7
                ACI only  min        C       11.72    19.88        7
            ACI + scheme mean        C       54.28    60.63        7
            ACI + scheme  min        C       11.72    19.88        7
         ACI + is_credit mean        E       60.02    60.02        3
         ACI + is_credit  min        B       16.40    19.88        3
ACI + scheme + is_credit mean        E       60.02    60.02        3
ACI + scheme + is_credit  min        B       16.40    19.88        3

Matches E:13.57:
Empty DataFrame
Columns: [variant, mode, best_aci, best_total, E_total, E_rules]
Index: []
